# New analysis 02 — Linear Regression and Random Forest on each pre-release indicator

**Question:** how much does each pre-release indicator, *on its own*, tell us about how many
owners a Steam game will reach? And how much do they add up to when combined?

**Setup**
- **Data:** `data/processed/steam_prerelease_features.csv.gz` from notebook 01 (labeled games
  released ≤ 2024-06-30, pre-release features only).
- **Target:** `log10(owners)` using the midpoint of the SteamSpy owner range, **measured
  relative to other games released in the same year** (Section 2 shows why). The log is needed
  because owners span 10k to 150M; on the raw scale a handful of blockbusters dominate every
  metric (the same Wii Sports problem as the old Phase 1).
- **Split: chronological.** Train on games released through 2021, test on 2022 – mid-2024. This
  is the honest deployment setting: the model never sees the future.
- **Models:** Linear Regression (the parent paper's model) and Random Forest, fit separately on
  each indicator group, then on groups combined.
- **Metrics on the test set:** R², RMSE and MAE in log10 units (an MAE of 0.3 ≈ off by a factor of
  2), Spearman rank correlation, and the ROC-AUC of the predicted value for separating hits
  (≥100k owners) from non-hits.

In [1]:
import json
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
DATA_DIR = os.path.join("..", "data", "processed")
OUT_DIR = os.path.join("..", "outputs")
TRAIN_END = pd.Timestamp("2021-12-31")
pd.set_option("display.width", 150)

df = pd.read_csv(os.path.join(DATA_DIR, "steam_prerelease_features.csv.gz"), parse_dates=["release_dt"])
groups_saved = pd.read_json(os.path.join(DATA_DIR, "feature_groups.json"), typ="series")
print(df.shape)

(76727, 88)


## 1. Preprocessing

- Heavy-tailed counts get a `log1p` version. Random Forest does not care, but Linear Regression
  would otherwise be dominated by a few extreme values.
- Missing values: `lead_days` (re-releases, <1%) is filled with the **training** median plus a
  missing flag; outcome-based studio history is filled with 0 plus a "has history" flag, so
  "no track record" stays distinguishable from "a track record of zero".

In [2]:
for c in ["n_languages", "n_audio_languages", "n_screenshots", "short_desc_len", "dev_prior_games",
          "pub_prior_games", "same_week_releases", "same_genre_month_releases", "achievements"]:
    df[f"log_{c}"] = np.log1p(df[c].fillna(0))

train_mask = df["release_dt"] <= TRAIN_END
df["lead_missing"] = df["log_lead_days"].isna().astype(int)
df["log_lead_days"] = df["log_lead_days"].fillna(df.loc[train_mask, "log_lead_days"].median())

df["has_dev_history"] = df["dev_prior_hit_rate"].notna().astype(int)
df["has_pub_history"] = df["pub_prior_hit_rate"].notna().astype(int)
for c in ["dev_prior_hit_rate", "dev_prior_max_log_owners", "pub_prior_hit_rate", "pub_prior_max_log_owners"]:
    df[c] = df[c].fillna(0)

df["short_description"] = df["short_description"].fillna("")
print(f"Train (<= {TRAIN_END.year}): {train_mask.sum():,} games | hit rate {df.loc[train_mask, 'is_hit'].mean():.1%}")
print(f"Test  (2022 - mid-2024): {(~train_mask).sum():,} games | hit rate {df.loc[~train_mask, 'is_hit'].mean():.1%}")

Train (<= 2021): 48,129 games | hit rate 13.8%
Test  (2022 - mid-2024): 28,598 games | hit rate 5.4%


## 2. Why the target is measured relative to the release year

The test period has a much lower hit rate than the training period. Two reasons, neither of which
has anything to do with a game's quality:
- **Time on sale.** SteamSpy counts owners up to the 2025 snapshot, so a 2016 game has had ~9
  years to sell and a 2023 game ~2.
- **Market growth.** Steam releases went from ~4k games/year (2016) to ~20k (2024), so the
  average game gets a smaller share.

In [3]:
by_year = df.groupby("year").agg(games=("appid", "size"), mean_log10_owners=("log_owners_mid", "mean"),
                                 hit_rate=("is_hit", "mean"))
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(by_year.loc[2010:].index, by_year.loc[2010:, "mean_log10_owners"], marker="o")
ax.axvline(TRAIN_END.year + 0.5, color="grey", ls="--", label="train | test")
ax.set_ylabel("mean log10(owners)")
ax.set_title("Average owners by release year")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "02_owners_by_release_year.png"), dpi=150)

raw_tr, raw_te = df.loc[train_mask, "log_owners_mid"], df.loc[~train_mask, "log_owners_mid"]
print(f"Predicting the training mean on the test period, raw log10(owners): "
      f"R2 = {r2_score(raw_te, np.full(len(raw_te), raw_tr.mean())):.3f}")
by_year.loc[2014:].round(3)

Predicting the training mean on the test period, raw log10(owners): R2 = -0.222


,games,mean_log10_owners,hit_rate
year,,,
2014,1485,4.900,0.393
2015,2454,4.667,0.273
2016,4024,4.534,0.208
2017,5747,4.351,0.124
2018,7305,4.262,0.084
2019,7082,4.238,0.082
2020,8615,4.262,0.087
2021,9497,4.246,0.077
2022,10010,4.207,0.059


Against the raw target, just predicting the training average scores R² ≈ −0.2 on the test years,
and every single-indicator R² comes out negative. That measures the downward drift, not the
indicators.

**Fix:** use `rel_log_owners` = log10(owners) − mean log10(owners) of all labeled games released
in the same year. The question becomes *"how does this game do compared with the other games
released that year?"* Each year falls entirely in train or entirely in test, so no labels cross
the split. Hit ROC-AUC keeps the original ≥100k definition, which only needs a ranking within the
test period.

In [4]:
df["rel_log_owners"] = df["log_owners_mid"] - df.groupby("year")["log_owners_mid"].transform("mean")
y = df["rel_log_owners"].values
is_hit = df["is_hit"].values

train, test = df[train_mask], df[~train_mask]
y_tr, y_te = y[train_mask.values], y[~train_mask.values]
hit_te = is_hit[~train_mask.values]
print(f"Relative target: train mean {y_tr.mean():.3f}, test mean {y_te.mean():.3f}, sd {y.std():.3f}")

Relative target: train mean 0.000, test mean 0.000, sd 0.495


## 3. The indicators

Each row below is one *indicator* — some are a single column, some are a small group of columns
that describe the same thing (e.g. all genre flags). The last two groups are kept apart because
of known caveats from notebook 01.

In [5]:
genre_cols = [c for c in df.columns if c.startswith("genre_")]
cat_cols = [c for c in groups_saved["PRE_RELEASE"] if c.startswith("cat_")]

INDICATORS = {
    "Price":                   ["log_price", "is_free"],
    "Languages":               ["log_n_languages", "log_n_audio_languages", "has_english"],
    "Screenshots":             ["log_n_screenshots"],
    "Platforms (Mac/Linux)":   ["n_platforms", "mac", "linux"],
    "Required age":            ["required_age"],
    "Release timing":          ["year", "month"],
    "Website / support email": ["has_website", "has_support_email"],
    "Description length":      ["log_short_desc_len", "log_detailed_desc_len"],
    "Publisher arrangement":   ["self_published", "n_developers"],
    "Studio experience (count)": ["log_dev_prior_games", "log_pub_prior_games"],
    "Store-page lead time":    ["log_lead_days", "lead_missing"],
    "Launch competition":      ["log_same_week_releases", "log_same_genre_month_releases"],
    "Title features":          ["title_words", "has_sequel_marker", "has_subtitle"],
    "Genres":                  genre_cols,
    "Categories":              cat_cols,
    "Description text (TF-IDF)": "TEXT",
}
CAVEATED = {
    "Studio track record (outcome-based) *": ["dev_prior_hit_rate", "dev_prior_max_log_owners", "has_dev_history",
                                              "pub_prior_hit_rate", "pub_prior_max_log_owners", "has_pub_history"],
    "Uncertain store features *": ["log_achievements", "cat_Steam_Workshop", "cat_Remote_Play_any"],
}
pd.DataFrame(
    [(k, "TF-IDF -> 100 SVD components" if v == "TEXT" else f"{len(v)}: " + ", ".join(v[:4]) + (" ..." if len(v) > 4 else ""))
     for k, v in {**INDICATORS, **CAVEATED}.items()],
    columns=["indicator", "columns"],
)

,indicator,columns
0,Price,"2: log_price, is_free"
1,Languages,"3: log_n_languages, log_n_audio_languages, has..."
2,Screenshots,1: log_n_screenshots
3,Platforms (Mac/Linux),"3: n_platforms, mac, linux"
4,Required age,1: required_age
5,Release timing,"2: year, month"
6,Website / support email,"2: has_website, has_support_email"
7,Description length,"2: log_short_desc_len, log_detailed_desc_len"
8,Publisher arrangement,"2: self_published, n_developers"
9,Studio experience (count),"2: log_dev_prior_games, log_pub_prior_games"


### Description text

`short_description` is turned into TF-IDF word/bigram features and compressed to 100
components with truncated SVD, so both models see a dense, fixed-size input. **The vectorizer and
SVD are fit on the training years only**, so vocabulary from 2022+ games cannot leak into
training.

In [6]:
tfidf = TfidfVectorizer(max_features=20_000, min_df=5, ngram_range=(1, 2), stop_words="english", sublinear_tf=True)
svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
text_tr = svd.fit_transform(tfidf.fit_transform(train["short_description"]))
text_te = svd.transform(tfidf.transform(test["short_description"]))
TEXT_COLS = [f"text_svd_{i}" for i in range(text_tr.shape[1])]
train = pd.concat([train.reset_index(drop=True), pd.DataFrame(text_tr, columns=TEXT_COLS)], axis=1)
test = pd.concat([test.reset_index(drop=True), pd.DataFrame(text_te, columns=TEXT_COLS)], axis=1)
INDICATORS["Description text (TF-IDF)"] = TEXT_COLS
print(f"TF-IDF vocabulary: {len(tfidf.vocabulary_):,} terms -> {len(TEXT_COLS)} components "
      f"({svd.explained_variance_ratio_.sum():.1%} of variance)")

TF-IDF vocabulary: 20,000 terms -> 100 components (10.3% of variance)


## 4. Fit both models on each indicator

In [7]:
def make_models():
    return {
        "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
        "Random Forest": RandomForestRegressor(
            n_estimators=200, min_samples_leaf=20, max_features=0.5,
            n_jobs=-1, random_state=RANDOM_STATE,
        ),
    }


def evaluate(pred):
    constant = np.ptp(pred) == 0
    return {
        "R2": r2_score(y_te, pred),
        "RMSE": mean_squared_error(y_te, pred) ** 0.5,
        "MAE": mean_absolute_error(y_te, pred),
        "Spearman": np.nan if constant else spearmanr(y_te, pred).statistic,
        "Hit_ROC_AUC": roc_auc_score(hit_te, pred),
    }


def run(name, cols, kind):
    rows = []
    for model_name, model in make_models().items():
        model.fit(train[cols], y_tr)
        rows.append({"indicator": name, "kind": kind, "model": model_name, "n_features": len(cols),
                     **evaluate(model.predict(test[cols]))})
    return rows


baseline = {"indicator": "Baseline (predict training mean)", "kind": "baseline", "model": "-", "n_features": 0,
            **evaluate(np.full(len(y_te), y_tr.mean()))}

rows = [baseline]
for name, cols in INDICATORS.items():
    rows += run(name, cols, "single")
for name, cols in CAVEATED.items():
    rows += run(name, cols, "single (caveated)")
single = pd.DataFrame(rows)
single.round(3)

,indicator,kind,model,n_features,R2,RMSE,MAE,Spearman,Hit_ROC_AUC
0,Baseline (predict training mean),baseline,-,0,0.000,0.407,0.280,NaN,0.500
1,Price,single,Linear Regression,2,0.036,0.400,0.271,0.132,0.749
2,Price,single,Random Forest,2,0.046,0.398,0.245,0.105,0.741
3,Languages,single,Linear Regression,3,0.023,0.403,0.267,0.179,0.746
4,Languages,single,Random Forest,3,0.083,0.390,0.252,0.197,0.762
5,Screenshots,single,Linear Regression,1,0.025,0.402,0.270,0.131,0.673
6,Screenshots,single,Random Forest,1,0.025,0.402,0.269,0.132,0.673
7,Platforms (Mac/Linux),single,Linear Regression,3,-0.006,0.409,0.274,0.005,0.518
8,Platforms (Mac/Linux),single,Random Forest,3,-0.006,0.409,0.274,0.005,0.517
9,Required age,single,Linear Regression,1,0.032,0.401,0.271,0.075,0.533


### Side by side: test R² and hit ROC-AUC per indicator

In [8]:
wide = single[single["kind"] != "baseline"].pivot_table(
    index="indicator", columns="model", values=["R2", "Hit_ROC_AUC"])
wide = wide.sort_values(("R2", "Random Forest"))

fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)
ypos = np.arange(len(wide))
for ax, metric, ref in [(axes[0], "R2", 0.0), (axes[1], "Hit_ROC_AUC", 0.5)]:
    ax.barh(ypos - 0.2, wide[(metric, "Linear Regression")], height=0.4, label="Linear Regression", color="#4C72B0")
    ax.barh(ypos + 0.2, wide[(metric, "Random Forest")], height=0.4, label="Random Forest", color="#55A868")
    ax.axvline(ref, color="black", lw=0.8)
    ax.set_title(f"Test {metric.replace('_', ' ')} (single indicator)")
axes[0].set_yticks(ypos)
axes[0].set_yticklabels(wide.index)
axes[0].set_xlabel("R² on year-relative log10(owners); 0 = predicting the mean")
axes[1].set_xlabel("ROC-AUC for hit (>=100k owners); 0.5 = chance")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "02_per_indicator_r2_auc.png"), dpi=150)
print("Saved outputs/02_per_indicator_r2_auc.png  (* = caveated, see notebook 01)")
wide.round(3).iloc[::-1]

Saved outputs/02_per_indicator_r2_auc.png  (* = caveated, see notebook 01)


Hit_ROC_AUC                              R2              
model                                 Linear Regression Random Forest Linear Regression Random Forest
indicator                                                                                            
Studio track record (outcome-based) *             0.755         0.762             0.165         0.200
Categories                                        0.775         0.778             0.086         0.099
Languages                                         0.746         0.762             0.023         0.083
Uncertain store features *                        0.741         0.750             0.054         0.082
Description text (TF-IDF)                         0.705         0.694             0.047         0.051
Price                                             0.749         0.741             0.036         0.046
Required age                                      0.533         0.533             0.032         0.034
Website / support email                           0.657         0.661             0.025         0.027
Screenshots                                       0.673         0.673             0.025         0.025
Publisher arrangement                             0.627         0.627             0.023         0.022
Studio experience (count)                         0.636         0.643             0.021         0.018
Description length                                0.691         0.660             0.032         0.010
Store-page lead time                              0.767         0.746             0.052         0.007
Title features                                    0.548         0.562             0.002         0.002
Release timing                                    0.506         0.513             0.000        -0.002
Platforms (Mac/Linux)                             0.518         0.517            -0.006        -0.006
Genres                                            0.651         0.652            -0.014        -0.017
Launch competition                                0.473         0.533            -0.001        -0.023

## 5. Combined models

Indicators are not independent (bigger studios price higher, localize more, and put pages up
earlier), so single-indicator scores cannot simply be added. Fit on groups combined, building up
in layers:
1. **All store-page indicators** (everything above except text),
2. **+ description text**,
3. **+ outcome-based studio track record** (caveat: measured at the 2025 snapshot),
4. **+ uncertain store features** (may be added after launch).

In [9]:
store = sorted({c for k, v in INDICATORS.items() if k != "Description text (TF-IDF)" for c in v})
layers = {
    "1. All store-page indicators": store,
    "2. + description text": store + TEXT_COLS,
    "3. + studio track record *": store + TEXT_COLS + CAVEATED["Studio track record (outcome-based) *"],
    "4. + uncertain store features *": store + TEXT_COLS + CAVEATED["Studio track record (outcome-based) *"]
                                       + CAVEATED["Uncertain store features *"],
}
combo_rows = [baseline]
for name, cols in layers.items():
    combo_rows += run(name, cols, "combined")
combined = pd.DataFrame(combo_rows)
combined.round(3)

,indicator,kind,model,n_features,R2,RMSE,MAE,Spearman,Hit_ROC_AUC
0,Baseline (predict training mean),baseline,-,0,0.000,0.407,0.280,NaN,0.500
1,1. All store-page indicators,combined,Linear Regression,67,0.113,0.384,0.275,0.279,0.832
2,1. All store-page indicators,combined,Random Forest,67,0.235,0.356,0.233,0.293,0.844
3,2. + description text,combined,Linear Regression,167,0.119,0.382,0.276,0.295,0.840
4,2. + description text,combined,Random Forest,167,0.223,0.359,0.237,0.283,0.844
5,3. + studio track record *,combined,Linear Regression,173,0.188,0.367,0.265,0.343,0.859
6,3. + studio track record *,combined,Random Forest,173,0.278,0.346,0.222,0.298,0.857
7,4. + uncertain store features *,combined,Linear Regression,176,0.200,0.364,0.262,0.344,0.862
8,4. + uncertain store features *,combined,Random Forest,176,0.289,0.344,0.220,0.302,0.861


In [10]:
cw = combined[combined["kind"] == "combined"].pivot_table(index="indicator", columns="model", values="R2")
ax = cw.plot(kind="bar", figsize=(9, 4.5), color=["#4C72B0", "#55A868"], rot=0)
ax.set_ylabel("Test R² on year-relative log10(owners)")
ax.set_title("Combined models, chronological test (2022 - mid-2024)")
ax.set_xticklabels([t.get_text().replace(". ", ".\n", 1) for t in ax.get_xticklabels()], fontsize=8)
for p in ax.patches:
    ax.annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "02_combined_models_r2.png"), dpi=150)
print("Saved outputs/02_combined_models_r2.png")

Saved outputs/02_combined_models_r2.png


## 6. Which indicators matter *inside* the combined model?

Group permutation importance on the test set for the Random Forest from layer 2 (all
leakage-free indicators): shuffle all columns of one indicator together and measure how much
test R² drops. Unlike the single-indicator scores in Section 4, this measures what each indicator
adds *given all the others*.

In [11]:
cols2 = layers["2. + description text"]
rf2 = make_models()["Random Forest"].fit(train[cols2], y_tr)
base_r2 = r2_score(y_te, rf2.predict(test[cols2]))
rng = np.random.default_rng(RANDOM_STATE)

perm_rows = []
for name, cols in INDICATORS.items():
    drops = []
    for _ in range(3):
        shuffled = test[cols2].copy()
        idx = rng.permutation(len(shuffled))
        shuffled[cols] = shuffled[cols].values[idx]
        drops.append(base_r2 - r2_score(y_te, rf2.predict(shuffled)))
    perm_rows.append({"indicator": name, "R2_drop_when_shuffled": np.mean(drops)})
perm = pd.DataFrame(perm_rows).sort_values("R2_drop_when_shuffled")

plt.figure(figsize=(8, 6))
plt.barh(perm["indicator"], perm["R2_drop_when_shuffled"], color="#8172B2")
plt.xlabel("drop in test R² when the indicator is shuffled")
plt.title(f"Group permutation importance (Random Forest, layer 2, test R² = {base_r2:.3f})")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "02_permutation_importance.png"), dpi=150)
print("Saved outputs/02_permutation_importance.png")
perm.iloc[::-1].round(4)

Saved outputs/02_permutation_importance.png


,indicator,R2_drop_when_shuffled
1,Languages,0.1250
0,Price,0.0451
10,Store-page lead time,0.0398
15,Description text (TF-IDF),0.0119
14,Categories,0.0081
11,Launch competition,0.0020
4,Required age,0.0018
9,Studio experience (count),0.0014
8,Publisher arrangement,0.0011
3,Platforms (Mac/Linux),0.0008


In [12]:
single.to_csv(os.path.join(OUT_DIR, "02_single_indicator_metrics.csv"), index=False)
combined.to_csv(os.path.join(OUT_DIR, "02_combined_model_metrics.csv"), index=False)
perm.to_csv(os.path.join(OUT_DIR, "02_permutation_importance.csv"), index=False)
print("Saved metric tables to outputs/")

Saved metric tables to outputs/


## Notes for the write-up

- Scores are on a **future** test period (2022 – mid-2024), so they are what a model trained on
  history would actually achieve on new releases.
- The target is relative to the release year, so `Release timing` can only capture seasonality
  (month), not the long-run downward trend.
- The target is binned (SteamSpy ranges), so even a perfect model could not reach R² = 1 against
  the true owner count; R² is relative to the bin midpoints.
- Indicators marked * carry the caveats from notebook 01: studio track record uses owner counts
  measured in 2025, and the "uncertain" store features may be added after launch. Their gains
  should be reported separately from the leakage-free layers 1–2.